# 31~39 Tool use with Claude

## 33_Tool function

In [8]:
!python --version

Python 3.14.5


In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
from datetime import datetime, timedelta

def get_current_datetime(
    date_format="%Y-%m-%d %H:%M:%S"
):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [5]:
date = get_current_datetime()
print(f"date: {date}")

date2 = get_current_datetime("%H:%M")
print(f"date2: {date2}")


date: 2026-08-04 10:10:57
date2: 10:47


In [6]:
get_current_datetime("")

ValueError: date_format cannot be empty

## 34_Tool Schema

In [10]:
from datetime import datetime, timedelta

def get_current_datetime(
    date_format="%Y-%m-%d %H:%M:%S"
):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_shema =     {
    "name": "get_current_datetime",
    "description": "Returns the current local date and time of the machine running this tool, formatted as a string. Use this whenever you need to know 'now' — for timestamping, computing relative dates (e.g. 'next Friday'), or answering questions about the current date or time. Do not use it for past or future dates, or for time in other time zones: the result always reflects the host's local system clock with no timezone conversion and no timezone offset in the output. The date_format parameter is a Python strftime format string that controls the output layout only; it does not change which moment is returned. If omitted, the format defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2026-08-04 09:30:00'). Passing an empty string raises an error.",
    "input_schema": {
        "type": "object",
        "properties": {
        "date_format": {
            "type": "string",
            "minLength": 1,
            "default": "%Y-%m-%d %H:%M:%S",
            "description": "Python strftime format string, e.g. '%Y-%m-%d' for date only, '%H:%M' for time only, '%Y-%m-%dT%H:%M:%S' for ISO-like output. Omit this parameter unless the user or task requires a specific layout. Must not be empty."
        }
        },
        "required": [],
        "additionalProperties": False
    },
    "input_examples": [
        {},
        { "date_format": "%Y-%m-%d" },
        { "date_format": "%Y年%m月%d日 %H時%M分" }
    ]
}

In [13]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current local date and time of the machine running this tool, formatted as a string. Use this whenever you need to know 'now' — for timestamping, computing relative dates (e.g. 'next Friday'), or answering questions about the current date or time. Do not use it for past or future dates, or for time in other time zones: the result always reflects the host's local system clock with no timezone conversion and no timezone offset in the output. The date_format parameter is a Python strftime format string that controls the output layout only; it does not change which moment is returned. If omitted, the format defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2026-08-04 09:30:00'). Passing an empty string raises an error.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "minLength": 1,
                "default": "%Y-%m-%d %H:%M:%S",
                "description": "Python strftime format string, e.g. '%Y-%m-%d' for date only, '%H:%M' for time only, '%Y-%m-%dT%H:%M:%S' for ISO-like output. Omit this parameter unless the user or task requires a specific layout. Must not be empty."
            }
        },
        "required": [],
        "additionalProperties": False
    },
    "input_examples": [
        {},
        { "date_format": "%Y-%m-%d" },
        { "date_format": "%Y年%m月%d日 %H時%M分" }
    ]
})

## 35_Handling message blocks
## 36_Sending_tool_result

In [49]:
messages = []

messages.append({
    "role" : "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})


In [50]:
response = client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [51]:
response

Message(id='msg_011CdhagWU3zsY84iFQvDGQv', container=None, content=[ToolUseBlock(id='toolu_01AcyPP5fta9A7DJtD2UiXde', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1036, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [52]:
response.content

[ToolUseBlock(id='toolu_01AcyPP5fta9A7DJtD2UiXde', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]

In [54]:
messages.append({"role": "assistant", "content": response.content})

In [55]:
tool_result = get_current_datetime(**response.content[0].input)
tool_result

'19:52:56'

In [63]:
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": tool_result,
            "is_error" : False
        }
    ]})
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01AcyPP5fta9A7DJtD2UiXde', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01AcyPP5fta9A7DJtD2UiXde',
    'content': '19:52:56',
    'is_error': False}]}]

In [64]:
final_response = client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    tools=[get_current_datetime_schema]
)
final_response

Message(id='msg_011CdhatgYNNCQn76E779uKw', container=None, content=[TextBlock(citations=None, text='The exact time is **19:52:56** (7:52:56 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1116, output_tokens=23, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

## 37_Multi-turn conversation with Claude

In [66]:
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role" : "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role" : "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


In [84]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature" : temperature,
        "stop_sequences" : stop_sequences
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message


In [68]:
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

## 38-39_Implementation_multiple_turns

In [80]:
def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


In [86]:
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError("Invalid Tool name.")

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type" : "tool_result",
                "tool_use_id" : tool_request.id,
                "content" : json.dumps(tool_output),
                "is_error" : False
            }
        except Exception as e:
            tool_result_block = {
                "type" : "tool_result",
                "tool_use_id" : tool_request.id,
                "content" : f"Error: {e}",
                "is_error" : True
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [87]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema
        ])
        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [ ]:
messages = []
add_user_message(messages, "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.")

response = run_conversation(messages)
response

----
Setting the following reminder for 2050-06-27T00:00:00:
Doctor's appointment
----


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='I need to first calculate the date that is 177 days after Jan 1st, 2050, and then set a reminder for you.', type='text'),
   ToolUseBlock(id='toolu_01FaWG6oqhLVPwkgaeWjkwUX', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01FaWG6oqhLVPwkgaeWjkwUX',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050 at midnight:", type='text'),
   ToolUseBlock(id='toolu_01EFyVZNP8771zk4CjX5cKtw', caller=DirectCaller(type='direct'), input={'c

: 